# Overview:

We're working with TLE (Two-Line Element Set) data.

TLE data will help us calculate the position + velocity over time.

Source: Celestrak

Dataset explanation:

Line 1 (Used for timing) :

- ID
- Launch info
- Time (epoch)
- Drag termm

Line 2 (Used for orbit calculation):

- Inclination / orbit tilt
- RAAN / orbit orientation
- Eccentricity / shape of orbit
- Argument of perigee
- Mean anomaly - position in orbit
- Mean motion - speed

# 1) Loading dataset

In [16]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/combined_tle_dataset.csv")
df.head()

,object_id,line1,line2,source
0,FY1C_001,1 25730U 99025A 26125.18059819 .00002636 0...,2 25730 98.8639 198.1396 0010510 23.9231 336...,Fengyun-1C
1,FY1C_002,1 29733U 99025X 26124.61711778 .00001187 0...,2 29733 99.2091 163.9268 0565528 231.7505 286...,Fengyun-1C
2,FY1C_003,1 29734U 99025Y 26124.72095970 .00001979 0...,2 29734 99.1715 36.2763 0616183 43.7835 346...,Fengyun-1C
3,FY1C_004,1 29735U 99025Z 26124.19038905 .00000556 0...,2 29735 99.3877 20.4567 0696613 219.4877 135...,Fengyun-1C
4,FY1C_005,1 29736U 99025AA 26124.42942706 .00000720 0...,2 29736 99.4193 281.6117 0723013 251.9880 131...,Fengyun-1C


In [17]:
print(df.columns)

Index(['object_id', 'line1', 'line2', 'source'], dtype='object')


# 2) Data Check


In [18]:
print(df.iloc[0])

"""Make sure:
line1 starts with 1
line2 starts with 2
To confirm that the dataset is valid"""

object_id                                             FY1C_001
line1        1 25730U 99025A   26125.18059819  .00002636  0...
line2        2 25730  98.8639 198.1396 0010510  23.9231 336...
source                                              Fengyun-1C
Name: 0, dtype: object


'Make sure:\nline1 starts with 1\nline2 starts with 2\nTo confirm that the dataset is valid'

# **Main TLE Module**

# 3) Creating trackable debris objects from data

In [19]:
!pip install skyfield

In [20]:
#the load module handles time syystems
#the EarthSatellite creates debris objects
from skyfield.api import load, EarthSatellite

#creating a time scale object (basically a clock for orbital calcullations)
#debris change position over time
#skyfield needs current time, future time and simulation time
ts = load.timescale()

#to store the debris objects
debris = []

for i, row in df.iterrows():
    deb = EarthSatellite(row['line1'], row['line2'], row['object_id'], ts)
    '''Converts line 1 and line 2 into:
    - orbital parameters
    - motion model
    - trackable object
    using the SGP4 orbital propagation model internally.'''
    debris.append(deb)

print("Loaded:", len(debris))

Loaded: 2560


Basically creating a mathematical object that can predict debris position in space using the orbital data of a debris object.

# 4) Testing one object

In [21]:
#get the current time
t = ts.now()
deb = debris[0]

#calculating the object position at time t
pos = deb.at(t).position.km
print(pos)

[-5915.29579062 -3525.53592455 -2024.12266594]


Gives us the 3D Cartesian Coordinates (x, y, z)

Unit: kilometers (relative to earth's center)


Coordinate Breakdown:

- x	= left-right position

- y =	forward-back position

- z =	up-down position

Together they define exact location in space

The SGP4 propagation model computes the 3D Cartesian coordinates of debris objects at a specified time using TLE orbital parameters.

This is not live GPS tracking because TLEs are predictive orbital models.

TLE describes motion. We use it to calculate where the object should be now.

# 5) Generate trajectory

First step to becoming a tracking and simulation system instead of just a position calculator.

Creating a trajectory (path of an object through space over time)

In [22]:
#creating multiple timestamps
#because we want to know where is the debris object every minute
times = ts.utc(2026, 5, 1, range(0, 60))  #60 minutes

trajectory = []

deb = debris[0]

#calculating coordinates through each timestamp
for t in times:
    pos = deb.at(t).position.km
    trajectory.append(pos)

#displaying the coordinates for the first 5 minutes
trajectory[:5]

[array([5027.70724624, 1971.93522509, 4729.28499231]),
 array([-6865.86277586, -1829.26757056, -1025.23015914]),
 array([ 6400.0328571 ,  1059.94940332, -3108.27870194]),
 array([-3738.69238251,    65.96327398,  6115.21328722]),
 array([ -164.45445001, -1175.53783817, -7093.12807456])]

Mathematically, the system predicts how the object moves based on orbital mechanics (at each minute) .

# 6) Save trajectories

In [23]:
data = []

#generating trajectories for multiple objects
for deb in debris[:10]:  #keeping it small!
    for t in times:
        pos = deb.at(t).position.km

        data.append({
            "object_id": deb.name,
            "time": str(t),
            "x": pos[0],
            "y": pos[1],
            "z": pos[2]
        })

df_traj = pd.DataFrame(data)
df_traj.head()

,object_id,time,x,y,z
0,FY1C_001,<Time tt=2461161.500800741>,5027.707246,1971.935225,4729.284992
1,FY1C_001,<Time tt=2461161.5424674074>,-6865.862776,-1829.267571,-1025.230159
2,FY1C_001,<Time tt=2461161.584134074>,6400.032857,1059.949403,-3108.278702
3,FY1C_001,<Time tt=2461161.625800741>,-3738.692383,65.963274,6115.213287
4,FY1C_001,<Time tt=2461161.6674674074>,-164.454450,-1175.537838,-7093.128075


In [24]:
df_traj.to_csv("trajectories.csv", index=False)

# 7) Download the file

In [25]:
from google.colab import files
#files.download("trajectories.csv")